# Ingest and format the knowledge graph (kg.txt and ratings)

In [ ]:
import pandas as pd

# Load Ratings
ratings = pd.read_csv('./ml-1m/ratings.dat', sep='::', engine='python', 
                      names=['UserID', 'MovieID', 'Rating', 'Timestamp'])

# Treat all interactions as positive reward (any rating from 1-5) to optimize for engagement
interactions = ratings[['UserID', 'MovieID']].copy()

# Format as head, relation, tail
interactions['Head'] = 'user_' + interactions['UserID'].astype(str)
interactions['Relation'] = 'user.interact.movie'
interactions['Tail'] = 'movie_' + interactions['MovieID'].astype(str)

# load knowledge graph
kg = pd.read_csv('kg.txt', sep='\t', names=['Head', 'Relation', 'Tail'])

kg['Head'] = kg['Head'].astype(str)
kg['Tail'] = 'entity_' + kg['Tail'].astype(str)

interaction_t = interactions[['Head', 'Relation', 'Tail']]

graph = pd.concat([interaction_t, kg], ignore_index=True)



print(ratings)
print(kg)


         UserID  MovieID  Rating  Timestamp
0             1     1193       5  978300760
1             1      661       3  978302109
2             1      914       3  978301968
3             1     3408       4  978300275
4             1     2355       5  978824291
...         ...      ...     ...        ...
1000204    6040     1091       1  956716541
1000205    6040     1094       5  956704887
1000206    6040      562       5  956704746
1000207    6040     1096       4  956715648
1000208    6040     1097       4  956715569

[1000209 rows x 4 columns]
       Head            Relation         Tail
0       749    film.film.writer  entity_2347
1      1410  film.film.language  entity_2348
2      1037  film.film.language  entity_2348
3      1088    film.film.writer  entity_2349
4      1391  film.film.language  entity_2348
...     ...                 ...          ...
20777  2308    film.film.writer  entity_4284
20778   869  film.film.language  entity_2348
20779  1953     film.film.genre  entity

In [ ]:
# add reverge edges
graph = pd.concat([graph, graph.rename(columns={'Head': 'Tail', 'Tail': 'Head'})], ignore_index=True)
# self loops
graph = pd.concat([graph, pd.DataFrame({'Head': graph['Head'], 'Relation': 'self_loop', 'Tail': graph['Head']})], ignore_index=True)

# Convert strings to ids for neural network
entities = pd.unique(graph[['Head', 'Tail']].values.ravel('K'))
entity_to_id = {entity: i for i, entity in enumerate(sorted(entities))}

relations = pd.unique(graph['Relation'])
relation_to_id = {relation: i for i, relation in enumerate(sorted(relations))}

graph['Head_ID'] = graph['Head'].map(entity_to_id)
graph['Tail_ID'] = graph['Tail'].map(entity_to_id)
graph['Relation_ID'] = graph['Relation'].map(relation_to_id)

numerical_graph = graph[['Head_ID', 'Relation_ID', 'Tail_ID']].values

print(graph)

                Head             Relation         Tail  Head_ID  Tail_ID  \
0             user_1  user.interact.movie   movie_1193    10714     7197   
1             user_1  user.interact.movie    movie_661    10714    10382   
2             user_1  user.interact.movie    movie_914    10714    10623   
3             user_1  user.interact.movie   movie_3408    10714     9511   
4             user_1  user.interact.movie   movie_2355    10714     8382   
...              ...                  ...          ...      ...      ...   
4083959  entity_4284            self_loop  entity_4284     4300     4300   
4083960  entity_2348            self_loop  entity_2348     2446     2446   
4083961  entity_2362            self_loop  entity_2362     2459     2459   
4083962  entity_5417            self_loop  entity_5417     5420     5420   
4083963  entity_2403            self_loop  entity_2403     2497     2497   

         Relation_ID  
0                  8  
1                  8  
2                 